In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Funções do Agente
# MAGIC
# MAGIC Este notebook cria funções que simulam as ferramentas do agente.
# MAGIC
# MAGIC As funções leem e escrevem nas tabelas transacionais Delta do app.
# MAGIC Futuramente, essas mesmas operações podem ser expostas para um agente
# MAGIC real usando Databricks Agents, LangChain, MLflow ou ferramentas customizadas.

# COMMAND ----------

import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

APP_USERS_TABLE = f"{CATALOG}.{SCHEMA}.app_users"
APP_PROFILES_TABLE = f"{CATALOG}.{SCHEMA}.app_profiles"
APP_JOB_POSTINGS_TABLE = f"{CATALOG}.{SCHEMA}.app_job_postings"
APP_SAVED_JOBS_TABLE = f"{CATALOG}.{SCHEMA}.app_saved_jobs"
APP_APPLICATIONS_TABLE = f"{CATALOG}.{SCHEMA}.app_applications"
APP_INTERVIEW_NOTES_TABLE = f"{CATALOG}.{SCHEMA}.app_interview_notes"
APP_CONTACTS_TABLE = f"{CATALOG}.{SCHEMA}.app_contacts"
MATCH_SCORES_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_match_scores"

DEMO_USER_ID = "demo_user_001"

print("=" * 70)
print("JOBFLOW AI — FUNÇÕES DO AGENTE")
print("=" * 70)
print(f"Usuário demo: {DEMO_USER_ID}")
print(f"Jobs table: {APP_JOB_POSTINGS_TABLE}")
print(f"Saved jobs table: {APP_SAVED_JOBS_TABLE}")
print(f"Applications table: {APP_APPLICATIONS_TABLE}")
print(f"Interview notes table: {APP_INTERVIEW_NOTES_TABLE}")
print(f"Horário UTC: {datetime.now(timezone.utc).isoformat()}")
print("=" * 70)

# COMMAND ----------

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Funções auxiliares

# COMMAND ----------

def utc_now_naive() -> datetime:
    return datetime.now(timezone.utc).replace(tzinfo=None)


def require_job_exists(job_id: str) -> None:
    count_value = (
        spark.table(APP_JOB_POSTINGS_TABLE)
        .where(F.col("job_id") == job_id)
        .count()
    )

    if count_value == 0:
        raise ValueError(f"job_id não encontrado: {job_id}")


def require_application_exists(application_id: str, user_id: str) -> None:
    count_value = (
        spark.table(APP_APPLICATIONS_TABLE)
        .where(F.col("application_id") == application_id)
        .where(F.col("user_id") == user_id)
        .count()
    )

    if count_value == 0:
        raise ValueError(
            f"application_id não encontrado para este usuário: {application_id}"
        )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Ferramenta de leitura: buscar melhores vagas

# COMMAND ----------

def buscar_vagas_recomendadas(
    user_id: str = DEMO_USER_ID,
    limite: int = 10,
):
    """
    Retorna vagas ranqueadas pelo score de compatibilidade.
    """

    df = (
        spark.table(MATCH_SCORES_TABLE)
        .where(F.col("user_id") == user_id)
        .select(
            "job_id",
            "match_score",
            "match_tier",
            "job_title",
            "company_name",
            "job_location",
            "remote_type",
            "salary_min",
            "salary_max",
            "matched_skills",
            "missing_skills",
            "match_explanation",
            "job_url",
            "apply_url",
        )
        .orderBy(F.col("match_score").desc(), F.col("job_title"))
        .limit(limite)
    )

    return df


display(buscar_vagas_recomendadas(limite=10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Ferramenta de leitura: detalhes da vaga

# COMMAND ----------

def obter_detalhes_vaga(job_id: str):
    """
    Retorna os detalhes principais de uma vaga.
    """

    require_job_exists(job_id)

    df = (
        spark.table(APP_JOB_POSTINGS_TABLE)
        .where(F.col("job_id") == job_id)
        .select(
            "job_id",
            "source_system",
            "source_job_id",
            "job_title",
            "company_name",
            "job_location",
            "remote_type",
            "published_at",
            "tags_text",
            "salary_min",
            "salary_max",
            "salary_currency",
            "job_description",
            "apply_url",
            "job_url",
            "is_active",
            "data_quality_score",
        )
    )

    return df


top_job_id = buscar_vagas_recomendadas(limite=1).first()["job_id"]
display(obter_detalhes_vaga(top_job_id))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Ferramenta de escrita: salvar vaga

# COMMAND ----------

def salvar_vaga(
    job_id: str,
    user_id: str = DEMO_USER_ID,
    priority: str = "medium",
    notes: str | None = None,
):
    """
    Salva uma vaga para o usuário.
    Se a vaga já estiver salva, substitui o registro anterior.
    """

    require_job_exists(job_id)

    saved_job_id = f"{user_id}_{job_id}"

    spark.sql(
        f"""
        DELETE FROM {APP_SAVED_JOBS_TABLE}
        WHERE user_id = '{user_id}'
          AND job_id = '{job_id}'
        """
    )

    schema = T.StructType(
        [
            T.StructField("saved_job_id", T.StringType(), False),
            T.StructField("user_id", T.StringType(), False),
            T.StructField("job_id", T.StringType(), False),
            T.StructField("priority", T.StringType(), True),
            T.StructField("notes", T.StringType(), True),
        ]
    )

    row = [
        {
            "saved_job_id": saved_job_id,
            "user_id": user_id,
            "job_id": job_id,
            "priority": priority,
            "notes": notes,
        }
    ]

    df = (
        spark.createDataFrame(row, schema)
        .withColumn("saved_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
    )

    df.write.mode("append").saveAsTable(APP_SAVED_JOBS_TABLE)

    return saved_job_id


saved_job_id = salvar_vaga(
    job_id=top_job_id,
    priority="high",
    notes="Salva via função do agente para demonstração.",
)

print(f"Vaga salva: {saved_job_id}")

display(
    spark.table(APP_SAVED_JOBS_TABLE)
    .where(F.col("saved_job_id") == saved_job_id)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Ferramenta de escrita: criar aplicação

# COMMAND ----------

def criar_aplicacao(
    job_id: str,
    user_id: str = DEMO_USER_ID,
    stage: str = "applied",
    notes: str | None = None,
):
    """
    Cria uma aplicação para uma vaga.
    Se já existir aplicação para user_id + job_id, substitui a anterior.
    """

    allowed_stages = [
        "saved",
        "applied",
        "interviewing",
        "offer",
        "rejected",
        "withdrawn",
    ]

    if stage not in allowed_stages:
        raise ValueError(f"stage inválido: {stage}. Valores aceitos: {allowed_stages}")

    require_job_exists(job_id)

    application_id = f"{user_id}_{job_id}_application"

    spark.sql(
        f"""
        DELETE FROM {APP_APPLICATIONS_TABLE}
        WHERE user_id = '{user_id}'
          AND job_id = '{job_id}'
        """
    )

    schema = T.StructType(
        [
            T.StructField("application_id", T.StringType(), False),
            T.StructField("user_id", T.StringType(), False),
            T.StructField("job_id", T.StringType(), False),
            T.StructField("stage", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
            T.StructField("notes", T.StringType(), True),
        ]
    )

    row = [
        {
            "application_id": application_id,
            "user_id": user_id,
            "job_id": job_id,
            "stage": stage,
            "status": "active",
            "notes": notes,
        }
    ]

    df = (
        spark.createDataFrame(row, schema)
        .withColumn(
            "applied_at",
            F.when(F.lit(stage) == "applied", F.current_timestamp())
            .otherwise(F.lit(None).cast("timestamp"))
        )
        .withColumn("last_activity_at", F.current_timestamp())
        .withColumn("next_followup_at", F.expr("current_timestamp() + INTERVAL 7 DAYS"))
        .withColumn("created_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
    )

    df.write.mode("append").saveAsTable(APP_APPLICATIONS_TABLE)

    return application_id


application_id = criar_aplicacao(
    job_id=top_job_id,
    stage="applied",
    notes="Aplicação criada via função do agente.",
)

print(f"Aplicação criada: {application_id}")

display(
    spark.table(APP_APPLICATIONS_TABLE)
    .where(F.col("application_id") == application_id)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Ferramenta de escrita: atualizar etapa da aplicação

# COMMAND ----------

def atualizar_etapa_aplicacao(
    application_id: str,
    new_stage: str,
    user_id: str = DEMO_USER_ID,
    notes: str | None = None,
):
    """
    Atualiza a etapa de uma aplicação.
    Como estamos em Delta simples, fazemos delete + insert com os dados atualizados.
    """

    allowed_stages = [
        "saved",
        "applied",
        "interviewing",
        "offer",
        "rejected",
        "withdrawn",
    ]

    if new_stage not in allowed_stages:
        raise ValueError(
            f"new_stage inválido: {new_stage}. Valores aceitos: {allowed_stages}"
        )

    require_application_exists(application_id, user_id)

    current_df = (
        spark.table(APP_APPLICATIONS_TABLE)
        .where(F.col("application_id") == application_id)
        .where(F.col("user_id") == user_id)
        .limit(1)
    )

    current_row = current_df.first()

    updated_notes = notes if notes is not None else current_row["notes"]

    updated_schema = T.StructType(
        [
            T.StructField("application_id", T.StringType(), False),
            T.StructField("user_id", T.StringType(), False),
            T.StructField("job_id", T.StringType(), False),
            T.StructField("stage", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
            T.StructField("applied_at", T.TimestampType(), True),
            T.StructField("created_at", T.TimestampType(), True),
            T.StructField("notes", T.StringType(), True),
        ]
    )

    updated_rows = [
        {
            "application_id": current_row["application_id"],
            "user_id": current_row["user_id"],
            "job_id": current_row["job_id"],
            "stage": new_stage,
            "status": current_row["status"],
            "applied_at": current_row["applied_at"],
            "created_at": current_row["created_at"],
            "notes": updated_notes,
        }
    ]

    updated_df = (
        spark.createDataFrame(updated_rows, updated_schema)
        .withColumn("last_activity_at", F.current_timestamp())
        .withColumn("next_followup_at", F.expr("current_timestamp() + INTERVAL 7 DAYS"))
        .withColumn("updated_at", F.current_timestamp())
        .select(
            "application_id",
            "user_id",
            "job_id",
            "stage",
            "status",
            "applied_at",
            "last_activity_at",
            "next_followup_at",
            "notes",
            "created_at",
            "updated_at",
        )
    )

    spark.sql(
        f"""
        DELETE FROM {APP_APPLICATIONS_TABLE}
        WHERE application_id = '{application_id}'
          AND user_id = '{user_id}'
        """
    )

    updated_df.write.mode("append").saveAsTable(APP_APPLICATIONS_TABLE)

    return application_id


atualizar_etapa_aplicacao(
    application_id=application_id,
    new_stage="interviewing",
    notes="Movida para interviewing via função do agente.",
)

display(
    spark.table(APP_APPLICATIONS_TABLE)
    .where(F.col("application_id") == application_id)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Ferramenta de escrita: adicionar nota de entrevista

# COMMAND ----------

def adicionar_nota_entrevista(
    application_id: str,
    round_type: str,
    note_text: str,
    user_id: str = DEMO_USER_ID,
):
    """
    Adiciona uma nota de entrevista vinculada a uma aplicação.
    """

    require_application_exists(application_id, user_id)

    application_row = (
        spark.table(APP_APPLICATIONS_TABLE)
        .where(F.col("application_id") == application_id)
        .where(F.col("user_id") == user_id)
        .first()
    )

    note_id = str(uuid.uuid4())

    schema = T.StructType(
        [
            T.StructField("note_id", T.StringType(), False),
            T.StructField("application_id", T.StringType(), False),
            T.StructField("user_id", T.StringType(), False),
            T.StructField("job_id", T.StringType(), True),
            T.StructField("round_type", T.StringType(), True),
            T.StructField("note_text", T.StringType(), True),
        ]
    )

    row = [
        {
            "note_id": note_id,
            "application_id": application_id,
            "user_id": user_id,
            "job_id": application_row["job_id"],
            "round_type": round_type,
            "note_text": note_text,
        }
    ]

    df = (
        spark.createDataFrame(row, schema)
        .withColumn("created_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
    )

    df.write.mode("append").saveAsTable(APP_INTERVIEW_NOTES_TABLE)

    return note_id


note_id = adicionar_nota_entrevista(
    application_id=application_id,
    round_type="recruiter_screen",
    note_text="Nota demo: revisar requisitos de Python, SQL e experiência remota.",
)

print(f"Nota criada: {note_id}")

display(
    spark.table(APP_INTERVIEW_NOTES_TABLE)
    .where(F.col("note_id") == note_id)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Ferramenta de leitura: listar aplicações

# COMMAND ----------

def listar_aplicacoes(user_id: str = DEMO_USER_ID):
    """
    Lista aplicações do usuário com dados da vaga.
    """

    applications_df = spark.table(APP_APPLICATIONS_TABLE).alias("a")
    jobs_df = spark.table(APP_JOB_POSTINGS_TABLE).alias("j")

    df = (
        applications_df
        .join(jobs_df, F.col("a.job_id") == F.col("j.job_id"), "left")
        .where(F.col("a.user_id") == user_id)
        .select(
            F.col("a.application_id"),
            F.col("a.stage"),
            F.col("a.status"),
            F.col("a.applied_at"),
            F.col("a.last_activity_at"),
            F.col("a.next_followup_at"),
            F.col("j.job_title"),
            F.col("j.company_name"),
            F.col("j.remote_type"),
            F.col("j.job_url"),
        )
        .orderBy(F.col("a.updated_at").desc())
    )

    return df


display(listar_aplicacoes())

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Ferramenta de leitura: aplicações paradas

# COMMAND ----------

def encontrar_aplicacoes_paradas(
    user_id: str = DEMO_USER_ID,
    stale_days: int = 5,
):
    """
    Lista aplicações sem atualização recente.
    """

    applications_df = spark.table(APP_APPLICATIONS_TABLE).alias("a")
    jobs_df = spark.table(APP_JOB_POSTINGS_TABLE).alias("j")

    df = (
        applications_df
        .join(jobs_df, F.col("a.job_id") == F.col("j.job_id"), "left")
        .where(F.col("a.user_id") == user_id)
        .where(F.col("a.status") == "active")
        .withColumn(
            "days_since_last_activity",
            F.datediff(F.current_date(), F.to_date(F.col("a.last_activity_at")))
        )
        .where(F.col("days_since_last_activity") >= stale_days)
        .select(
            F.col("a.application_id"),
            F.col("a.stage"),
            F.col("a.last_activity_at"),
            F.col("a.next_followup_at"),
            F.col("days_since_last_activity"),
            F.col("j.job_title"),
            F.col("j.company_name"),
        )
        .orderBy(F.col("days_since_last_activity").desc())
    )

    return df


display(encontrar_aplicacoes_paradas(stale_days=0))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Validação final das ferramentas

# COMMAND ----------

validation_rows = [
    ("recommended_jobs", buscar_vagas_recomendadas(limite=100).count()),
    ("saved_jobs", spark.table(APP_SAVED_JOBS_TABLE).where(F.col("user_id") == DEMO_USER_ID).count()),
    ("applications", spark.table(APP_APPLICATIONS_TABLE).where(F.col("user_id") == DEMO_USER_ID).count()),
    ("interview_notes", spark.table(APP_INTERVIEW_NOTES_TABLE).where(F.col("user_id") == DEMO_USER_ID).count()),
]

validation_df = spark.createDataFrame(validation_rows, ["artifact", "records"])

display(validation_df)

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: FUNÇÕES DO AGENTE CONCLUÍDAS")
print("=" * 70)
print("Ferramentas de leitura:")
print("- buscar_vagas_recomendadas")
print("- obter_detalhes_vaga")
print("- listar_aplicacoes")
print("- encontrar_aplicacoes_paradas")
print()
print("Ferramentas de escrita:")
print("- salvar_vaga")
print("- criar_aplicacao")
print("- atualizar_etapa_aplicacao")
print("- adicionar_nota_entrevista")
print("=" * 70)